<a href="https://colab.research.google.com/github/cbishop4/MSE7530/blob/main/ClassNotebooks/F2026/Lecture13_InClass_20261021.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MSE/CHE 7530/5995: October 21st, 2026  
Topics: AFM Height/Phase Correlation, Image Processing

## Imports & Github Cloning

In [ ]:
!pip install uv -q
!uv pip install pySPM --system -q

In [ ]:
import pySPM as spm
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import LogNorm
from matplotlib.pyplot import cm
import re

from mpl_toolkits.axes_grid1.anchored_artists import AnchoredSizeBar
import matplotlib.font_manager as fm

In [ ]:
# Download the AFM data from the public repo
!git clone https://github.com/cbishop4/Alahe2026.git

In [ ]:
!git clone https://github.com/hyperspy/hyperspy-demos.git

## Height & Phase Correlation in AFM
Here we will just plot the height and phase next to each other; if time, we will talk conceptually about ways to quantify this. It can become quite complex, and full details are out of the scope of this class. However, you can come to office hours if you're interested and want to learn more.

In [ ]:
def pull_params(filename,print_explicit=False):
  scan = spm.Bruker(filename) # method to load a Bruker file
  scan_size = scan.layers[0][b'Scan Size'] # returns raw string
  num_lines = scan.layers[0][b'Number of lines'] # returns raw string
  ss = float(re.findall(r'\d+(?:\.\d+)?', str(scan_size[0]))[0]) # uses regex to pull out the float from the scan size string
  nlines = int(re.findall(r'\d+', str(num_lines[0]))[0]) # uses regex to pull out the integer from the "number of lines" string
  if ss < 45.: # scan size in metadata is stored in microns
    ss_nm = ss*1000. # convert microns to nm
  else:
    ss_nm = ss # number is large enough that we assume it was recorded in nm
  px_nm = ss_nm/nlines # divide scan size in nm by the number of lines (resolution)
  if print_explicit: # if we change it to True
    print(f'The scan size is {ss_nm} nm, there are {nlines} lines, and the pixel size is {px_nm} nm')
  return ss_nm, nlines, px_nm

### Plotting Next to each other.
I'm going to do a separate cell for each rate. An example for the first file is provided; you can copy-paste to do it for each.

In [ ]:
f = '/content/Alahe2026/AFMdata/0p08nms/MAH26_01.030'
fig, ax = plt.subplots(1,2,figsize=(10,5))
scan = spm.Bruker(f)

channel1 = scan.get_channel('Height')
channel_corr = channel1.correct_lines(inline=False)
ht = channel_corr.pixels
channel1 = scan.get_channel('Phase')
channel_corr = channel1.correct_lines(inline=False)
phz = channel_corr.pixels
ss_nm, nlines, px_nm = pull_params(f,print_explicit=False)
cb = ax[0].imshow(ht)
ax[0].set_title('Height')
fig.colorbar(cb, ax=ax[0], location='bottom',label='Height (nm)')


cb = ax[1].imshow(phz)
ax[1].set_title('Phase')
fig.colorbar(cb, ax=ax[1], location='bottom',label='Phase Angle (deg)')

for i in range(2):
  scalebar = AnchoredSizeBar(ax[i].transData,
                            round(500 / px_nm), '500 nm', 'lower center',
                            pad=0.1,
                            color='white',
                            frameon=False,
                            size_vertical=3)
  ax[i].add_artist(scalebar)

I'm also going to zoom in on this so you can look a little more closely.

In [ ]:
fig, ax = plt.subplots(1,2,figsize=(10,5))
scan = spm.Bruker(f)

channel1 = scan.get_channel('Height')
channel_corr = channel1.correct_lines(inline=False)
ht = channel_corr.pixels
channel1 = scan.get_channel('Phase')
channel_corr = channel1.correct_lines(inline=False)
phz = channel_corr.pixels
ss_nm, nlines, px_nm = pull_params(f,print_explicit=False)
cb = ax[0].imshow(ht)
ax[0].set_title('Height')
fig.colorbar(cb, ax=ax[0], location='bottom',label='Height (nm)')


cb = ax[1].imshow(phz)
ax[1].set_title('Phase')
fig.colorbar(cb, ax=ax[1], location='bottom',label='Phase Angle (deg)')

for i in range(2):
  ax[i].set_xlim(200,400)
  ax[i].set_ylim(200,400)

In [ ]:
# your code here for repeats

Qualitatively, what do you see?

### Linecuts.
To decrease the amount of dimensions we need to visually process over, we can take linecuts out of the image and compare on a single axis.

In [ ]:
scan = spm.Bruker(f)

channel1 = scan.get_channel('Height')
channel_corr = channel1.correct_lines(inline=False)
ht = channel_corr.pixels
channel1 = scan.get_channel('Phase')
channel_corr = channel1.correct_lines(inline=False)
phz = channel_corr.pixels

fig, ax = plt.subplots(1,3,figsize=(15,5))
cb = ax[0].imshow(ht)
ax[0].set_title('Height')
fig.colorbar(cb, ax=ax[0], location='bottom',label='Height (nm)')
cb = ax[1].imshow(phz)
ax[1].set_title('Phase')
fig.colorbar(cb, ax=ax[1], location='bottom',label='Phase')
ax[0].axhline(250,color='red'); ax[1].axhline(250,color='orange')
ax[2].plot(ht[250,:],color='red',label='height');
ax[2].plot(phz[250,:],color='orange',label='phase')
ax[2].set_xlabel('Pixel')
ax[2].set_ylabel('Height/Phase')
ax[2].set_xlim(0,200)
ax[2].legend()

### Correlation plots.
We can also plot the height pixels vs. the phase pixels and see how correlated they are with a Pearson correlation coefficient.

In [ ]:
htf = ht.flatten()
phzf = phz.flatten()

plt.plot(htf[:5000],phzf[:5000],'o') # only plotting 5000 points because full million+ points kills the runtime

from scipy.stats import pearsonr as pr
res = pr(htf,phzf) # we can still calculate the pearson coefficient with the full set of data
print(res)
plt.xlabel('Height (nm)')
plt.ylabel('Phase angle (degrees)')

## Image Math - Addition and Subtraction

In [ ]:
!pip install hyperspy -q
import hyperspy.api as hs

In [ ]:
%matplotlib inline

In [ ]:
# load up the stack using hyperspy
import copy
s_stack = hs.load("/content/hyperspy-demos/image_data/datasets/stem_abf_data_imagestack.hspy")

This loads up a stack of STEM data. I'm going to write a couple unconventional lines of code to pull some things out so we can do them manually.

In [ ]:
uncorr = {}
for i in range(5):
  uncorr[i] = s_stack.__dict__['_data'][i]

That stack contained a series of micrographs; here we will plot them.

In [ ]:
fig, ax = plt.subplots(1,5,figsize=(15,3))

for i in uncorr.keys():
  cb = ax[i].imshow(uncorr[i],cmap='Grays_r',origin='lower')
  plt.colorbar(cb,ax=ax[i],location='top',label='Intensity (a.u.)')

We'll do the same thing but zoom in:

In [ ]:
fig, ax = plt.subplots(1,5,figsize=(15,3))

for i in uncorr.keys():
  cb = ax[i].imshow(uncorr[i],cmap='Grays_r',origin='lower')
  plt.colorbar(cb,ax=ax[i],location='top',label='Intensity (a.u.)')
  ax[i].set_xlim(0,50); ax[i].set_ylim(0,50)

What happens if you add these images together?

In [ ]:
# take this empty 2D array and add all of the images together
summed = np.zeros(uncorr[0].shape)
for i in uncorr.keys():
  summed = summed + uncorr[i]

In [ ]:
fig, ax = plt.subplots(1,2)
cb = ax[0].imshow(summed,cmap='Grays_r',origin='lower')
plt.colorbar(cb,ax=ax[0],location='top',label='Intensity (a.u.)')
cb = ax[1].imshow(summed,cmap='Grays_r',origin='lower')
plt.colorbar(cb,ax=ax[1],location='top',label='Intensity (a.u.)')

ax[1].set_xlim(0,100); ax[1].set_ylim(0,100)

What happened here?

We need to align the images. There is a built-in function to do this.

In [ ]:
#alignment_list = s_stack.align2D(roi=(250, 600, 250, 600), crop=True, reference='cascade', hanning=True, sobel=True, medfilter=True)
alignment_list = s_stack.align2D(crop=True, reference='cascade', hanning=True, sobel=True, medfilter=True)
# note that this fundamentally changes s_stack

In [ ]:
alignment_list

In [ ]:
corr = {}
for i in range(5):
  corr[i] = s_stack.__dict__['_data'][i]

In [ ]:
# note that the shape of these has changed
print(f'New shape is {corr[2].shape} compared to the unaligned shape of {uncorr[2].shape}')

In [ ]:
fig, ax = plt.subplots(1,5,figsize=(15,3))

for i in corr.keys():
  cb = ax[i].imshow(corr[i],cmap='Grays_r',origin='lower')
  plt.colorbar(cb,ax=ax[i],location='top',label='Intensity (a.u.)')
  ax[i].set_xlim(0,50); ax[i].set_ylim(0,50)

In [ ]:
summed_al = np.zeros(corr[0].shape)
for i in corr.keys():
  summed_al = summed_al + corr[i]

In [ ]:
fig, ax = plt.subplots(1,2)
cb = ax[0].imshow(summed_al,cmap='Grays_r',origin='lower')
plt.colorbar(cb,ax=ax[0],location='top',label='Intensity (a.u.)')
cb = ax[1].imshow(summed_al,cmap='Grays_r',origin='lower')
plt.colorbar(cb,ax=ax[1],location='top',label='Intensity (a.u.)')

ax[1].set_xlim(0,100); ax[1].set_ylim(0,100)

In [ ]:
# direct comparison of images before addition, added without alignment, and added with alignment:
fig, ax = plt.subplots(1,3,figsize=(15,5))
cb = ax[0].imshow(uncorr[0],cmap='Grays_r',origin='lower')
plt.colorbar(cb,ax=ax[0],location='bottom',label='Intensity (a.u.)')
ax[0].set_title('Single image')

cb = ax[1].imshow(summed,cmap='Grays_r',origin='lower')
plt.colorbar(cb,ax=ax[1],location='bottom',label='Intensity (a.u.)')
ax[1].set_title('Summed w/o alignment')

cb = ax[2].imshow(summed_al,cmap='Grays_r',origin='lower')
plt.colorbar(cb,ax=ax[2],location='bottom',label='Intensity (a.u.)')
ax[2].set_title('Summed w/ alignment')

for a in ax:
  a.set_xlim(0,100); a.set_ylim(0,100)

## Image Subtraction - Find the Difference

In [ ]:
!git clone https://github.com/cbishop4/MSE7530.git

In [ ]:
import cv2

In [ ]:
im1 = cv2.imread('/content/MSE7530/sampledata/HW6/CrystalVideo/frame0.jpg',0)
im2 = cv2.imread('/content/MSE7530/sampledata/HW6/CrystalVideo/frame50.jpg',0)
im3 = cv2.imread('/content/MSE7530/sampledata/HW6/CrystalVideo/frame125.jpg',0)
fig, ax = plt.subplots(1,3,figsize=(15,5))
ax[0].imshow(im1,cmap='Greys_r')
ax[1].imshow(im2,cmap='Greys_r')
ax[2].imshow(im3,cmap='Greys_r')

In [ ]:
plt.imshow(im2-im1,cmap='Greys_r')

In [ ]:
im1 = cv2.imread('/content/MSE7530/sampledata/HW6/CrystalVideo/frame0.jpg',0)
im2 = cv2.imread('/content/MSE7530/sampledata/HW6/CrystalVideo/frame10.jpg',0)
#im3 = cv2.imread('/content/MSE7530/sampledata/HW6/CrystalVideo/frame125.jpg',0)
fig, ax = plt.subplots(1,3,figsize=(15,5))
ax[0].imshow(im1,cmap='Greys_r')
ax[1].imshow(im2,cmap='Greys_r')
ax[2].imshow(im2- im1,cmap='Greys_r')

In [ ]:
im1 = cv2.imread('/content/MSE7530/sampledata/HW6/CrystalVideo/frame0.jpg',0)
im2 = cv2.imread('/content/MSE7530/sampledata/HW6/CrystalVideo/frame20.jpg',0)
#im3 = cv2.imread('/content/MSE7530/sampledata/HW6/CrystalVideo/frame125.jpg',0)
fig, ax = plt.subplots(1,3,figsize=(15,5))
ax[0].imshow(im1,cmap='Greys_r')
ax[1].imshow(im2,cmap='Greys_r')
ax[2].imshow(im2- im1,cmap='Greys_r')

Play around with image subtraction and find when the crystals start growing. What are the difficulties associated with this?